# SILVER_MUNICIPIO

Municípios do Brasil padronizados:
- código IBGE de 6 dígitos (padrão do DATASUS) e de 7 dígitos;
- **região de saúde** e **macrorregião** oficiais (só Alagoas; os demais estados ficam como `FORA DE ALAGOAS`);
- `TIPO_LOCALIDADE`: `CAPITAL` (Maceió), `INTERIOR` (demais municípios de AL) ou `OUTRA UF`.

**Origem:** `BRONZE_IBGE` + `SILVER_DOMINIO`.

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO SILVER_MUNICIPIO
    (COD_IBGE_6, COD_IBGE_7, NOME_MUNICIPIO, UF, REGIAO_SAUDE, MACRORREGIAO, TIPO_LOCALIDADE)
SELECT SUBSTR(i.COD_IBGE_7, 1, 6),
       i.COD_IBGE_7,
       UPPER(TRIM(i.NOME_MUNICIPIO)),
       i.UF,
       CASE WHEN i.UF = 'AL' THEN NVL(r.DESCRICAO, 'NAO INFORMADO') ELSE 'FORA DE ALAGOAS' END,
       CASE WHEN i.UF = 'AL' THEN NVL(m.DESCRICAO, 'NAO INFORMADO') ELSE 'FORA DE ALAGOAS' END,
       CASE WHEN i.UF <> 'AL' THEN 'OUTRA UF'
            WHEN SUBSTR(i.COD_IBGE_7, 1, 6) = '270430' THEN 'CAPITAL'
            ELSE 'INTERIOR' END
FROM BRONZE_IBGE i
LEFT JOIN SILVER_DOMINIO r ON r.TABELA = 'REGIAO_SAUDE' AND r.CODIGO = SUBSTR(i.COD_IBGE_7, 1, 6)
LEFT JOIN SILVER_DOMINIO m ON m.TABELA = 'MACRORREGIAO' AND m.CODIGO = SUBSTR(i.COD_IBGE_7, 1, 6)
"""

In [ ]:
recarregar("SILVER_MUNICIPIO", SQL, origem="BRONZE_IBGE")

In [ ]:
consultar("SELECT * FROM SILVER_MUNICIPIO FETCH FIRST 10 ROWS ONLY")